# BEIR NQ RARS vs PCA — Colab T4 + Drive

This notebook builds only the Stage 0–2 qrels-free package. It never extracts or reads `qrels/test.tsv`. Read `docs/beir_nq_colab_runbook.md` before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install the frozen user-space packages. Colab's CUDA-enabled torch is retained.
%pip -q install 'sentence-transformers==3.4.1' 'faiss-gpu-cu12==1.12.0' 'pytest>=8,<9'

In [ ]:
from pathlib import Path
import os, re, shutil, subprocess
import faiss, torch

REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/rars-beir-nq-confirmation-v1')
DESIGN_FREEZE_COMMIT = 'PASTE_FULL_40_HEX_COMMIT_HERE'

assert re.fullmatch(r'[0-9a-f]{40}', DESIGN_FREEZE_COMMIT), 'Paste the full Stage-0 commit hash.'
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
assert 'T4' in torch.cuda.get_device_name(0).upper(), torch.cuda.get_device_name(0)
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
assert free_gb >= 25, f'Need at least 25 GB free on Drive; found {free_gb:.1f} GB'
print('GPU:', torch.cuda.get_device_name(0))
print('Faiss:', getattr(faiss, '__version__', 'unknown'), 'GPUs:', faiss.get_num_gpus())
print('Drive free:', f'{free_gb:.1f} GB')

In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', 'beir-nq-confirmation', DESIGN_FREEZE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
assert head == DESIGN_FREEZE_COMMIT and not status
os.chdir(REPO)
print('Frozen checkout:', head)

In [ ]:
# Local synthetic gates; no NQ data or qrels are touched.
subprocess.run([
    'python', '-m', 'pytest', '-q',
    'tests/test_prepare_beir_nq_colab.py',
    'tests/test_train_select_beir_nq_sidecars.py',
    'tests/test_create_beir_nq_splits.py',
    'tests/test_nq_pre_qrels_freeze.py',
    'tests/test_beir_nq_freeze_helpers.py',
], check=True)

In [ ]:
def run_script(script, *args):
    command = ['python', str(REPO / script), *map(str, args)]
    print(' '.join(command))
    subprocess.run(command, check=True)

run_script(
    'scripts/prepare_beir_nq_colab.py', 'init',
    '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
    '--design-freeze-commit', DESIGN_FREEZE_COMMIT,
)

## Stage 1A — verified source and train-only extraction
The archive is verified, but `qrels/test.tsv` remains inside it and is not extracted.

In [ ]:
for command in ['download', 'extract-train-only', 'scan-corpus']:
    run_script(
        'scripts/prepare_beir_nq_colab.py', command,
        '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
    )

run_script(
    'scripts/create_beir_nq_train_validation_splits.py',
    '--queries', ARTIFACT_ROOT / 'stage1/data/nq/queries.jsonl',
    '--train-qrels', ARTIFACT_ROOT / 'stage1/data/nq/qrels/train.tsv',
    '--output-dir', ARTIFACT_ROOT / 'stage1/query_splits',
)

## Stage 1B — resumable document/query encoding
The first cell is long-running. If Colab disconnects, remount the same Drive, restore the same commit, and rerun it.

In [ ]:
run_script(
    'scripts/prepare_beir_nq_colab.py', 'encode-corpus',
    '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
    '--batch-size', 256, '--checkpoint-rows', 10000,
)

In [ ]:
run_script(
    'scripts/prepare_beir_nq_colab.py', 'encode-train-queries',
    '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
    '--batch-size', 512,
)

## Stage 1C — frozen IVF-PQ build
Training/addition uses a single T4. A CPU checkpoint is written every 250,000 added documents.

In [ ]:
run_script(
    'scripts/prepare_beir_nq_colab.py', 'build-index',
    '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
    '--add-batch-size', 50000, '--checkpoint-rows', 250000,
)

## Stage 2 — PCA/RARS fit, validation selection, and pre-qrels manifest

In [ ]:
run_script(
    'scripts/train_select_beir_nq_sidecars.py',
    '--artifact-root', ARTIFACT_ROOT,
    '--repo', REPO,
    '--search-batch-size', 256, '--exact-batch-size', 64,
    '--residual-batch-size', 20000, '--checkpoint-rows', 100000,
)

In [ ]:
run_script(
    'scripts/build_beir_nq_pre_qrels_manifest.py',
    '--artifact-root', ARTIFACT_ROOT, '--repo', REPO,
)
run_script(
    'scripts/validate_nq_pre_qrels_freeze.py',
    '--manifest', REPO / 'protocols/beir_nq_pre_qrels_manifest.json',
    '--artifact-root', ARTIFACT_ROOT, '--repo-root', REPO,
    '--verify-files',
)

In [ ]:
# Review these generated files. Do not open test qrels yet.
print(subprocess.check_output(['git', '-C', str(REPO), 'status', '--short'], text=True))
print((REPO / 'protocols/beir_nq_pre_qrels_manifest.json').read_text()[:4000])
print('NEXT: inspect, commit the generated manifest/configs, and record the full METHOD_FREEZE_COMMIT.')
print('No commit or push is performed by this notebook.')